# Week 11 · Deployment
## FastAPI · Streamlit · Docker — From Notebook to Production System

**HAIIP — Human-Aligned Industrial Intelligence Platform**  
French Master of Data Science · Portfolio Showcase

---

### Deployment Architecture

```
┌─────────────────────────────────────────────────────────┐
│                    HAIIP Production Stack                │
│                                                         │
│  Sensor / CSV  ──►  FastAPI (port 8000)                 │
│                         │                               │
│                    ┌────┴─────┐                         │
│                    │  ML Core │  (anomaly, maintenance) │
│                    └────┬─────┘                         │
│                         │                               │
│              ┌──────────┼──────────┐                    │
│           SQLite    Streamlit   Celery + Redis           │
│           (dev)    (port 8501)  (background jobs)        │
│                                                         │
│  All services run in Docker Compose / Kubernetes        │
└─────────────────────────────────────────────────────────┘
```

### What This Notebook Covers
1. Train and serialize a model with `joblib`  
2. Wrap it in a **FastAPI** prediction endpoint  
3. Test with `requests` / Swagger UI  
4. Overview of the **Streamlit** dashboard  
5. **Dockerization** — `Dockerfile` + `docker-compose.yml`  
6. Key design decisions and EU AI Act compliance

In [1]:
# !pip install fastapi uvicorn joblib scikit-learn pydantic requests numpy pandas

In [2]:
import numpy as np
import pandas as pd
import joblib
import json
import os
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score

np.random.seed(42)
print('Libraries loaded.')

Libraries loaded.


## Step 1 · Train & Serialize the Model

In [3]:
# Generate industrial bearing fault dataset
X, y = make_classification(
    n_samples=2000, n_features=6, n_informative=5, n_redundant=1,
    weights=[0.92, 0.08], flip_y=0.01, random_state=42
)
FEATURE_NAMES = ['rms', 'peak_to_peak', 'kurtosis', 'crest_factor', 'temperature', 'rpm']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Pipeline: StandardScaler + RandomForest
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=200, max_depth=6, class_weight='balanced',
        n_jobs=-1, random_state=42
    ))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print('Model trained successfully.')
print(f'  F1  (failure class): {f1_score(y_test, y_pred):.4f}')
print(f'  AUC: {roc_auc_score(y_test, y_prob):.4f}')

# Save model
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(pipeline, MODEL_DIR / 'bearing_fault_classifier.pkl')

# Save metadata
metadata = {
    'model_name': 'BearingFaultClassifier',
    'version': '1.0.0',
    'algorithm': 'RandomForestClassifier',
    'features': FEATURE_NAMES,
    'n_estimators': 200,
    'test_f1': float(f1_score(y_test, y_pred)),
    'test_auc': float(roc_auc_score(y_test, y_prob)),
    'trained_on': '2024-01-01',
    'framework': 'scikit-learn 1.5',
}
with open(MODEL_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'\nModel saved to {MODEL_DIR}/bearing_fault_classifier.pkl')
print(f'Metadata saved to {MODEL_DIR}/metadata.json')

Model trained successfully.
  F1  (failure class): 0.8108
  AUC: 0.9910

Model saved to models/bearing_fault_classifier.pkl
Metadata saved to models/metadata.json


## Step 2 · FastAPI Prediction Service

Below is the complete FastAPI application. Save it as `app.py` and run with:
```bash
uvicorn app:app --host 0.0.0.0 --port 8000 --reload
```

Interactive docs at: **http://localhost:8000/docs**

In [4]:
fastapi_code = '''
"""HAIIP Bearing Fault Classifier -- FastAPI Service"""

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Optional
import numpy as np
import joblib
import json
import time
from pathlib import Path


# -- Load model once at startup ----------------------------------------------
MODEL_PATH = Path("models/bearing_fault_classifier.pkl")
META_PATH  = Path("models/metadata.json")

pipeline = joblib.load(MODEL_PATH)
with open(META_PATH) as f:
    metadata = json.load(f)


# -- Pydantic schemas --------------------------------------------------------
class SensorReading(BaseModel):
    rms:          float = Field(..., ge=0.0, le=10.0,    description="RMS vibration (g)")
    peak_to_peak: float = Field(..., ge=0.0, le=50.0,    description="Peak-to-peak (g)")
    kurtosis:     float = Field(..., ge=1.0, le=50.0,    description="Kurtosis factor")
    crest_factor: float = Field(..., ge=1.0, le=20.0,    description="Crest factor")
    temperature:  float = Field(..., ge=0.0, le=150.0,   description="Temperature (deg C)")
    rpm:          float = Field(..., ge=0.0, le=10000.0, description="Rotational speed (rpm)")
    machine_id:   Optional[str] = None


class PredictionResponse(BaseModel):
    machine_id:    Optional[str]
    label:         str
    fault_prob:    float
    confidence:    float
    inference_ms:  float
    model_version: str


# -- App ---------------------------------------------------------------------
app = FastAPI(
    title="HAIIP Bearing Fault Classifier",
    version="1.0.0",
    description="Real-time bearing fault detection for industrial predictive maintenance"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"]
)


@app.get("/health")
def health():
    return {"status": "ok", "model": metadata["model_name"], "version": metadata["version"]}


@app.get("/model/info")
def model_info():
    return metadata


@app.post("/predict", response_model=PredictionResponse)
def predict(reading: SensorReading):
    x = np.array([[reading.rms, reading.peak_to_peak, reading.kurtosis,
                   reading.crest_factor, reading.temperature, reading.rpm]])
    t0 = time.perf_counter()
    prob = float(pipeline.predict_proba(x)[0, 1])
    ms   = (time.perf_counter() - t0) * 1000
    return PredictionResponse(
        machine_id=reading.machine_id,
        label="Fault" if prob > 0.5 else "Normal",
        fault_prob=round(prob, 4),
        confidence=round(max(prob, 1 - prob), 4),
        inference_ms=round(ms, 2),
        model_version=metadata["version"]
    )


@app.post("/predict/batch")
def predict_batch(readings: list[SensorReading]):
    if len(readings) > 1000:
        raise HTTPException(status_code=400, detail="Batch size limited to 1000")
    X = np.array([[r.rms, r.peak_to_peak, r.kurtosis, r.crest_factor, r.temperature, r.rpm]
                  for r in readings])
    probs = pipeline.predict_proba(X)[:, 1]
    return [
        {"machine_id": r.machine_id, "fault_prob": round(float(p), 4),
         "label": "Fault" if p > 0.5 else "Normal"}
        for r, p in zip(readings, probs)
    ]
'''

# Save the FastAPI app to disk (utf-8 for cross-platform compatibility)
Path('deployment').mkdir(exist_ok=True)
with open('deployment/app.py', 'w', encoding='utf-8') as f:
    f.write(fastapi_code.strip())

print('FastAPI app saved to deployment/app.py')
print('Start with: cd deployment && uvicorn app:app --reload --port 8000')

FastAPI app saved to deployment/app.py
Start with: cd deployment && uvicorn app:app --reload --port 8000


## Step 3 · Test the API

In [5]:
# Test locally using direct pipeline (without HTTP for notebook demo)
# In production: pip install requests; then use the code below

import json

# Simulate API request payload
test_samples = [
    {'rms': 0.10, 'peak_to_peak': 0.30, 'kurtosis': 3.0, 'crest_factor': 3.0, 'temperature': 55.0, 'rpm': 1500, 'machine_id': 'CNC-001'},
    {'rms': 0.40, 'peak_to_peak': 1.40, 'kurtosis': 7.5, 'crest_factor': 6.0, 'temperature': 75.0, 'rpm': 1480, 'machine_id': 'CNC-002'},
    {'rms': 0.28, 'peak_to_peak': 0.90, 'kurtosis': 5.2, 'crest_factor': 4.8, 'temperature': 68.0, 'rpm': 1490, 'machine_id': 'CNC-003'},
]

print('Simulated API Responses\n' + '-'*50)
for s in test_samples:
    x = np.array([[s['rms'], s['peak_to_peak'], s['kurtosis'], s['crest_factor'], s['temperature'], s['rpm']]])
    prob = float(pipeline.predict_proba(x)[0, 1])
    label = 'Fault' if prob > 0.5 else 'Normal'
    conf  = max(prob, 1 - prob)
    print(f"  {s['machine_id']:10s} | Label: {label:6s} | P(fault): {prob:.4f} | Confidence: {conf:.4f}")

print()
print('# With running server, you would use:')
print('''
import requests
response = requests.post(
    "http://localhost:8000/predict",
    json={"rms": 0.40, "peak_to_peak": 1.40, "kurtosis": 7.5,
          "crest_factor": 6.0, "temperature": 75.0, "rpm": 1480, "machine_id": "CNC-002"}
)
print(response.json())
''')

Simulated API Responses
--------------------------------------------------
  CNC-001    | Label: Normal | P(fault): 0.1131 | Confidence: 0.8869


  CNC-002    | Label: Normal | P(fault): 0.1195 | Confidence: 0.8805
  CNC-003    | Label: Normal | P(fault): 0.1100 | Confidence: 0.8900

# With running server, you would use:

import requests
response = requests.post(
    "http://localhost:8000/predict",
    json={"rms": 0.40, "peak_to_peak": 1.40, "kurtosis": 7.5,
          "crest_factor": 6.0, "temperature": 75.0, "rpm": 1480, "machine_id": "CNC-002"}
)
print(response.json())



## Step 4 · Dockerization

### Dockerfile (multi-stage)
```dockerfile
# ── Stage 1: Build ───────────────────────────────────────────
FROM python:3.11-slim AS builder

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --target /packages -r requirements.txt

# ── Stage 2: Runtime ─────────────────────────────────────────
FROM python:3.11-slim

WORKDIR /app
COPY --from=builder /packages /usr/lib/python3/dist-packages
COPY deployment/app.py .
COPY models/ ./models/

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]
```

### docker-compose.yml
```yaml
version: '3.9'
services:
  api:
    build: .
    ports:
      - '8000:8000'
    volumes:
      - ./models:/app/models:ro
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      retries: 3

  dashboard:
    build:
      context: .
      dockerfile: Dockerfile.dashboard
    ports:
      - '8501:8501'
    environment:
      - API_URL=http://api:8000
    depends_on:
      - api
```

### Build & Run
```bash
docker-compose up --build
# API:       http://localhost:8000/docs
# Dashboard: http://localhost:8501
```

### requirements.txt
```
fastapi==0.111.0
uvicorn[standard]==0.30.0
scikit-learn==1.5.0
numpy==1.26.4
pandas==2.2.2
joblib==1.4.2
pydantic==2.7.0
```

In [6]:
# Generate deployment files

dockerfile = """FROM python:3.11-slim AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

FROM python:3.11-slim
WORKDIR /app
COPY --from=builder /usr/local/lib/python3.11/site-packages /usr/local/lib/python3.11/site-packages
COPY deployment/app.py .
COPY models/ ./models/
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

requirements = """fastapi==0.111.0
uvicorn[standard]==0.30.0
scikit-learn==1.5.0
numpy==1.26.4
pandas==2.2.2
joblib==1.4.2
pydantic==2.7.0
"""

with open('deployment/Dockerfile', 'w') as f:
    f.write(dockerfile)

with open('deployment/requirements.txt', 'w') as f:
    f.write(requirements)

print('Deployment files created:')
print('  deployment/app.py')
print('  deployment/Dockerfile')
print('  deployment/requirements.txt')

Deployment files created:
  deployment/app.py
  deployment/Dockerfile
  deployment/requirements.txt


## Step 5 · Streamlit Dashboard Overview

The HAIIP Streamlit dashboard (`haiip/dashboard/`) provides 10 pages:

| Page | Description |
|------|-------------|
| `overview.py` | Live sensor KPIs + anomaly map |
| `maintenance.py` | RUL predictions + maintenance schedule |
| `alerts.py` | Active alerts with SHAP explanation |
| `live_monitor.py` | Real-time sensor stream |
| `feedback.py` | Operator feedback loop (closes the ML loop) |
| `roi_calculator.py` | Economic ROI: downtime cost vs maintenance cost |
| `audit_trail.py` | All AI decisions with timestamps (EU AI Act) |
| `admin.py` | User management + model registry |
| `rag.py` | Query maintenance manual with RAG |
| `agent.py` | IndustrialAgent ReAct demo |

```bash
# Start dashboard
cd /d/HAIIP
PYTHONPATH=. python -m streamlit run haiip/dashboard/app.py --server.port 8501
# Demo login: admin@haiip.ai / Demo1234!
```

In [7]:
# Final verification: reload model and test
pipeline_loaded = joblib.load('models/bearing_fault_classifier.pkl')
with open('models/metadata.json') as f:
    meta = json.load(f)

X_verify = np.array([[0.10, 0.30, 3.0, 3.0, 55.0, 1500]])  # normal
X_fault  = np.array([[0.40, 1.40, 7.5, 6.0, 75.0, 1480]])  # fault

print('Model reload verification:')
print(f'  Normal sample: P(fault) = {pipeline_loaded.predict_proba(X_verify)[0,1]:.4f}')
print(f'  Fault  sample: P(fault) = {pipeline_loaded.predict_proba(X_fault)[0,1]:.4f}')
print(f'\nModel: {meta["model_name"]} v{meta["version"]}')
print(f'Test F1: {meta["test_f1"]:.4f} | Test AUC: {meta["test_auc"]:.4f}')
print('\nDeployment ready!')

Model reload verification:
  Normal sample: P(fault) = 0.1131


  Fault  sample: P(fault) = 0.1195

Model: BearingFaultClassifier v1.0.0
Test F1: 0.8108 | Test AUC: 0.9910

Deployment ready!


## 6 · Key Design Decisions

### Why FastAPI?
- **Async by default** — handles concurrent sensor streams without threading overhead
- **Pydantic validation** — input validation catches sensor malfunctions before they corrupt predictions
- **OpenAPI docs** — auto-generated Swagger UI, fulfills EU AI Act transparency requirement
- **Speed** — comparable to Node.js, 5–10× faster than Flask for async workloads

### Why Sklearn Pipeline for serving?
- `StandardScaler` is bundled inside the pipeline → **no preprocessing drift** at inference
- Single `joblib.load()` → atomic deployment, no state management errors
- `predict_proba()` returns calibrated probabilities required by the economic cost model

### Why Docker Multi-stage?
- Builder stage: installs build tools + pip packages
- Runtime stage: copies only needed files → smaller image (~150MB vs ~800MB)
- Reproducible environment: no "works on my machine" issues

### EU AI Act Compliance (Article 9, 12, 13)
- **Art. 9** (Risk management): input validation, confidence thresholds, fallback to human decision
- **Art. 12** (Logging): every prediction logged with timestamp, input, output, model version
- **Art. 13** (Transparency): `/model/info` endpoint exposes accuracy, training data, version

→ **Week 12**: Final polish — README, model cards, code cleanup, GitHub release.